In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

DATA_CSV = Path('sample_data/wdi_world_slice.csv')
TARGET = 'SP.DYN.CBRT.IN'
FEATURES = {
  'SP.DYN.TFRT.IN': 'fertility_rate',
  'NY.GDP.PCAP.CD': 'gdp_per_capita_usd',
  'SP.DYN.LE00.IN': 'life_expectancy',
  'SP.URB.TOTL.IN.ZS': 'urban_population_pct',
  'SE.XPD.TOTL.GD.ZS': 'education_expenditure_pct_gdp',
  'SL.UEM.TOTL.ZS': 'unemployment_pct',
  'SP.POP.GROW': 'population_growth_pct',
  'SH.DYN.MORT': 'under5_mortality_per1000',
}
X_COLS = ['fertility_rate', 'gdp_per_capita_usd', 'log_gdp', 'life_expectancy', 'urban_population_pct', 'education_expenditure_pct_gdp', 'unemployment_pct', 'population_growth_pct', 'under5_mortality_per1000', 'health_index', 'urbanization_growth']

raw = pd.read_csv(DATA_CSV, low_memory=False)
years = [c for c in raw.columns if ' [YR' in c and c[:4].isdigit()]
long = raw.melt(id_vars=['Country Name', 'Country Code', 'Series Name', 'Series Code'], value_vars=years, var_name='year_col', value_name='value')
long['year'] = long['year_col'].str.slice(0, 4).astype(int)
long['value'] = pd.to_numeric(long['value'].replace('..', np.nan), errors='coerce')
long = long[long['Series Code'].isin([TARGET] + list(FEATURES.keys()))]
table = long.pivot_table(index='year', columns='Series Code', values='value', aggfunc='mean').reset_index()
table = table.rename(columns={TARGET: 'birth_rate', **FEATURES})
table['log_gdp'] = np.log1p(table['gdp_per_capita_usd'].fillna(0))
table['health_index'] = table['life_expectancy'] - table['under5_mortality_per1000'] / 10
table['urbanization_growth'] = table['urban_population_pct'] * (1 + table['population_growth_pct'].fillna(0) / 100)
table = table.dropna(subset=['birth_rate'])

X = table[X_COLS].fillna(table[X_COLS].median())
y = table['birth_rate']
scaler = StandardScaler()
Xs = scaler.fit_transform(X)
reg = Ridge().fit(Xs, y)
table['cluster_id'] = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(Xs[:, :4])
joblib.dump(reg, Path('models/regressor.joblib'))
meta = {'features': X_COLS, 'r2': float(reg.score(Xs, y))}
print(json.dumps(meta, indent=2))

In [ ]:
plt.scatter(table['year'], table['birth_rate'], c=table['cluster_id'])
plt.show()

In [ ]:
pd.Series(reg.coef_, index=X_COLS).plot(kind='barh')
plt.show()